In [ ]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(f"dirnames{dirname}   folders:{len(_)}  images:{len(filenames)} ")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
os.listdir("../input")

dataset

    train
        closed
        open
        
    test
        closed
        open
        

since our data is in the form required for image_data_from_directory , we will directy use this

# Libraries

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
# import 

In [ ]:
train_dir ="/kaggle/input/eyes-open-closed-dataset/dataset/train"
test_dir ="/kaggle/input/eyes-open-closed-dataset/dataset/test"

# Visualisation

In [ ]:
import os
import random
import matplotlib.pyplot as plt

def view_20random_image(train_dir):
    rows = 4
    cols = 5
    
    fig, axes = plt.subplots(rows, cols, figsize=(10, 10))
    fig.suptitle("20 Random Images from Dataset", fontsize=16)
    axes = axes.flatten()
    print(axes)

    for i in range(20):
        class_chosen = random.choice(os.listdir(train_dir))
        folder_path = os.path.join(train_dir, class_chosen)
        img_name = random.choice(os.listdir(folder_path))
        img_path = os.path.join(folder_path, img_name)
        
        img = plt.imread(img_path)
        ax = axes[i]
        ax.imshow(img)
        ax.set_title(class_chosen, fontsize=8)
        ax.axis("off")  
    

    plt.tight_layout()
    plt.show()


view_20random_image(train_dir)

In [ ]:
open_eyes = len( os.listdir( os.path.join(train_dir , "closed") ) )
closed_eyes=len( os.listdir( os.path.join(train_dir , "open") ) )
print("Number of closed eyes images" ,len( os.listdir( os.path.join(train_dir , "closed") ) ))
print("Number of open eyes images" ,len( os.listdir( os.path.join(train_dir , "open") ) ))

In [ ]:

# Data for the pie chart
labels = ['Closed Eyes', 'Open Eyes']
sizes = [open_eyes, closed_eyes]
colors = ['#ff9999','#66b3ff']
plt.figure(figsize=(3,3))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140, wedgeprops={'edgecolor': 'black'})
plt.title('Distribution of Open and Closed Eyes')
plt.show()


### once again looking at the folder structure


In [ ]:
for a,_,c in os.walk("/kaggle/input"):
    print(f"dir_name:{a} folders:{len(_)} images: {len(c)} ")

In [ ]:
data_dir ="/kaggle/input/eyes-open-closed-dataset/dataset"
train_dir ="/kaggle/input/eyes-open-closed-dataset/dataset/train"
test_dir="/kaggle/input/eyes-open-closed-dataset/dataset/test"

# Preprocessing


Doing preprocessing using image_dataset_from_directory

In [ ]:
BATCH_SIZE=32
IMG_SIZE=(224,224)
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(train_dir,
                                                                 batch_size=BATCH_SIZE,
                                                                 image_size=IMG_SIZE)


test_dataset = tf.keras.preprocessing.image_dataset_from_directory(test_dir,
                                                        
                                                                 batch_size=BATCH_SIZE,
                                                                 image_size=IMG_SIZE)



In [ ]:
#4634 training images in total and 818 testing images

#In one batch there are 32 images of shape (224,224,3) and corresponding to it 32 images with label as 0 or 1

### How a single batch looks like

In [ ]:
for image, label in train_dataset.take(1):
    print(image.shape)
    print(label.shape)


print("Num classes",train_dataset.class_names)
# 0 is for closed and 1 is for open

In [ ]:
print("Number of batches in trainig data" ,tf.data.experimental.cardinality(train_dataset))
print("Number of batches in test set",tf.data.experimental.cardinality(test_dataset))

### Splitting training into training + validation

In [ ]:
# There are 145 batches in training set and 26 in testing 
# taking out (1/10)th batches from the training set and make it validation


number_train_batches= tf.data.experimental.cardinality(train_dataset)
validation_dataset = train_dataset.take(number_train_batches//3)
train_dataset = train_dataset.skip(number_train_batches//3)

In [ ]:
print(f"Batches (Training) : {tf.data.experimental.cardinality(train_dataset)}")
print(f"Batches (Validation) : {tf.data.experimental.cardinality(validation_dataset)}")
print(f"Batches (Testing) : {tf.data.experimental.cardinality(test_dataset)}")

# Data Augmentation

Creating a data augmentation layer using keras Sequential API , which will fit directly into our model and the augmentation will happen only in case of training images not testing

1. Memory Efficient
2. Since augmentation happens during training , this augmentation layer can leverage the GPU , resulting in fast augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal'),
  # tf.keras.layers.RandomRotation(degrees=(90, -90), fill=(0,)),
    tf.keras.layers.RandomFlip('vertical'),
  tf.keras.layers.RandomZoom(0.2),
])

### Visualizing augmented image

In [ ]:

for image , _ in train_dataset.take(1):
  plt.figure(figsize=(10,10))
  first_image = image[0]
  label=_[0]
  class_name="closed"
  if label ==1:
      class_name="open"
  plt.figure(figsize=(3,3))
  plt.title(f"Original Image {class_name}")
  plt.axis("off")
  plt.imshow(first_image/255.)
  plt.show()
  print(first_image.shape)
  for i in range(9):
    ax=plt.subplot(3,3,i+1)
    augmented_image = data_augmentation(tf.expand_dims(first_image,0))
    plt.imshow(augmented_image[0]/255)
    plt.axis("off")

In [ ]:
for image ,label in train_dataset.take(1):
    print(label[0])
    print(image[0])


### Scaling the pixel values to  [-1,1] 

# Creating Feature extraction Model

Using MobileNetV2 as our base model

In [ ]:
base_model= tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    alpha=1.0,
    include_top=False,
    weights="imagenet"
)


In [ ]:
base_model.summary()

In [ ]:
layer_count=0
for layer in base_model.layers:
    layer_count+=1

print("In base Model number of layers ",layer_count)

Using Functional API to create a Feature extraction model


In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input
inputs = tf.keras.Input(shape=(224,224,3))


x=data_augmentation(inputs)
x=preprocess_input(x)
x=base_model(x,training=False)
x=tf.keras.layers.GlobalAveragePooling2D()(x)
x=tf.keras.layers.Dropout(0.2)(x)
x=tf.keras.layers.Dense(4,activation="relu")(x)
x = tf.keras.layers.Dropout(0.2)(x)

outputs=tf.keras.layers.Dense(1,activation='sigmoid')(x) 

model1 = tf.keras.Model(inputs,outputs)




In [ ]:
model1.summary()

Visualising input output shapes per layer

In [ ]:
for layer in model1.layers:
    print(layer ,layer.trainable)

In [ ]:
#compiling the model
model1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=[tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='accuracy')])

In [ ]:
# Running Feature extraction model for 5 epochs
initial_epochs=5


history1=model1.fit(train_dataset,
                  epochs=initial_epochs,
                  validation_data=validation_dataset)

In [ ]:
model1.evaluate(test_dataset)

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curves(history):
    """
    This function takes the training history object and plots the training and validation loss curves.
    
    Parameters:
    history (History object): The history object returned from model.fit()
    """
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    accuracy =history.history['accuracy']
    val_accuracy=history.history["val_accuracy"]
    plt.figure(figsize=(18, 6))
    plt.subplot(1,2,1)
    plt.plot(loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.grid(True)
    plt.ylabel('Loss')

    # plt.figure(figsize=(10, 6))
    plt.subplot(1,2,2)
    plt.plot(accuracy, label='Training Accurcay')
    plt.plot(val_accuracy, label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accurcy')
  
    

    plt.legend()

    plt.grid(True)
    plt.show()

plot_loss_curves(history1)

Makiing predictions on randomly chosen test data

In [ ]:
model1.evaluate(test_dataset)

# Prediction on one test batch using Model1

In [ ]:

image_batch, label_batch = test_dataset.as_numpy_iterator().next()
predictions = model1.predict_on_batch(image_batch).flatten()
predictions = tf.where(predictions < 0.5, 0, 1)
print('Predictions:\n', predictions.numpy())
print('Labels:\n', label_batch)

class_names =["closed","open"]

plt.figure(figsize=(10, 10))
for i in range(9):
  ax = plt.subplot(3, 3, i + 1)
  plt.imshow(image_batch[i].astype("uint8"))
  plt.title(class_names[predictions[i]])
  plt.axis("off")

# Confusion Matrix

In [ ]:
prediction = model1.predict(test_dataset)

In [ ]:
# Initialize empty lists to store the true labels and predictions
true_labels = []
predictions = []

# Iterate over the test dataset
for images, labels in test_dataset:
    # Get model predictions
    preds = model1.predict(images)
    preds=tf.squeeze(preds).numpy()
    # print(labels.shape)
    true_labels.extend(labels.numpy())
    predictions.extend(preds)

    for i in range(len(labels)):
        predicted_label = 1 if preds[i] >= 0.5 else 0
        if labels[i] == 1 and predicted_label == 0:
            print(f"Displaying image with actual label 1 but predicted label 0: Predicted {preds[i]}")

            # Display the image with proper normalization (assuming 0-255 pixel values)
            plt.imshow(images[i] / 255.)  # Normalize pixel values to 0-1 for display
            plt.axis("off")  # Hide axes
            plt.show()

rounded_predictions = []
for pred in predictions:
    if pred<0.5:
        rounded_predictions.append(0)
    else:
        rounded_predictions.append(1)
print(len(true_labels))
# print(rounded_predictions)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns


cm = confusion_matrix(true_labels, rounded_predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Closed", "Open"], yticklabels=["Closed", "Open"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
model1.summary()


In [ ]:
model1.save("eye-detection3.keras")